# Estudio para el examen — versión Jupyter Notebook

Este notebook toma como base `/home/runner/work/IA-AprendizajeAutomatico_Labs/IA-AprendizajeAutomatico_Labs/EstudioExamen/estudio_examen.py` y lo reorganiza para que sea más fácil estudiar, identificar qué bloque usar en el examen y ejecutar sólo la parte que necesites.

## Cómo usar este notebook

1. **Ejecuta primero la sección de imports y helpers**.
2. **Lee el árbol de decisión** para identificar si el ejercicio es de clasificación, regresión o MLP.
3. **Ejecuta únicamente el bloque del problema que te sirva**; no hace falta correr todo el notebook completo.
4. En cada bloque verás comentarios `>>> CAMBIAR` con los parámetros que normalmente debes adaptar en el examen.
5. Las celdas marcadas como **opcional** sólo sirven para visualizar resultados como curvas ROC o gráficos.

> Recomendación: durante el examen usa este notebook como guía de estudio y copia únicamente el bloque de código que resuelve el tipo de problema pedido.

## Árbol de decisión rápido

**¿Qué quieres predecir?**

- **Un número continuo** (precio, ventas, temperatura) → **Regresión**
  - Si la relación parece una recta → **Problema 4: Regresión lineal**
  - Si la relación tiene curvatura → **Problema 5: Regresión polinómica**
- **Una o varias categorías** → **Clasificación**
  - Una respuesta **sí/no** → **Problema 1: Binaria**
  - **Una sola clase entre varias** → **Problema 2: Multiclase**
  - **Varias etiquetas independientes a la vez** → **Problema 3: Multietiqueta**

Si el enunciado menciona **red neuronal** o **MLP**:

- Usa **Problema 6** si te sirve `sklearn`.
- Usa **Problema 7** si te exigen explícitamente **Keras/TensorFlow**.

Utilitarios complementarios:

- **Problema 8**: validación cruzada k-fold.
- **Problema 9**: curva ROC + AUC.

## 0. Imports y helpers comunes

**Ejecuta esta sección una sola vez al principio.**

Aquí se cargan todas las librerías y se definen funciones reutilizables:

- `reporte_clasificacion(...)`: imprime accuracy, precision, recall, F1 y AUC cuando aplica.
- `graficar_roc(...)`: dibuja la curva ROC para clasificación binaria.
- `RUTA_LAB3`: localiza automáticamente la carpeta de regresión para que los ejemplos con CSV funcionen mejor desde notebook.

In [ ]:

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, fetch_openml, make_classification
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, auc,
    mean_squared_error, r2_score,
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'estudio_examen.py').exists():
    ESTUDIO_DIR = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / 'EstudioExamen' / 'estudio_examen.py').exists():
    ESTUDIO_DIR = NOTEBOOK_DIR / 'EstudioExamen'
else:
    ESTUDIO_DIR = NOTEBOOK_DIR

REPO_ROOT = ESTUDIO_DIR.parent
RUTA_LAB3 = REPO_ROOT / 'Lab3_Regresion'

print(f'Directorio de estudio detectado: {ESTUDIO_DIR}')
print(f'Ruta de Lab3 detectada         : {RUTA_LAB3}')

def reporte_clasificacion(y_real, y_pred, y_score=None, etiqueta=''):
    print(f"\n--- Métricas {etiqueta} ---")
    print(f"  Accuracy : {accuracy_score(y_real, y_pred):.4f}")
    promedio = 'binary' if y_real.ndim == 1 and len(set(np.ravel(y_real))) == 2 else 'macro'
    print(f"  Precision: {precision_score(y_real, y_pred, average=promedio, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_real, y_pred, average=promedio, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y_real, y_pred, average=promedio, zero_division=0):.4f}")
    if y_score is not None:
        try:
            auc_v = roc_auc_score(y_real, y_score, multi_class='ovr', average='macro')
            print(f"  AUC      : {auc_v:.4f}")
        except ValueError:
            pass

def graficar_roc(y_real, y_score, titulo='Curva ROC'):
    fpr, tpr, _ = roc_curve(y_real, y_score)
    auc_v = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC = {auc_v:.4f}')
    plt.plot([0, 1], [0, 1], '--', color='gray', label='azar')
    plt.xlabel('Tasa de falsos positivos')
    plt.ylabel('Tasa de verdaderos positivos')
    plt.title(titulo)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    return auc_v

## 1. Clasificación binaria

**Úsalo cuando:** el problema sea de tipo **sí/no**, por ejemplo *“¿es un 5 o no?”*.

**Qué hace este bloque:**

- carga `digits` o `mnist`,
- convierte la salida original a una etiqueta binaria,
- divide en entrenamiento y prueba,
- entrena el modelo,
- calcula métricas y matriz de confusión.

**Qué cambiar normalmente:**

- `DIGITO_POSITIVO`
- `DATASET`
- `MODELO`

**Orden recomendado:**

1. Ejecuta la celda principal.
2. Si necesitas la ROC, ejecuta luego la celda opcional de gráfica.

In [ ]:
DIGITO_POSITIVO = 5      # >>> CAMBIAR: clase positiva
DATASET = 'digits'       # >>> CAMBIAR: 'digits' o 'mnist'
MODELO = 'sgd'           # >>> CAMBIAR: 'sgd' | 'knn' | 'logreg' | 'rf'

if DATASET == 'digits':
    data = load_digits()
    X, y = data.data, data.target
else:
    data = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X = data.data.astype(np.float32)
    y = data.target.astype(int)

y_bin = (y == DIGITO_POSITIVO).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.25, random_state=42, stratify=y_bin
)

if MODELO == 'sgd':
    clf = SGDClassifier(loss='log_loss', random_state=42, max_iter=1000)
elif MODELO == 'knn':
    clf = KNeighborsClassifier(n_neighbors=5)
elif MODELO == 'logreg':
    clf = LogisticRegression(solver='lbfgs', max_iter=1000)
elif MODELO == 'rf':
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
else:
    raise ValueError(f'Modelo desconocido: {MODELO}')

clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

if hasattr(clf, 'predict_proba'):
    y_score = clf.predict_proba(X_te)[:, 1]
elif hasattr(clf, 'decision_function'):
    y_score = clf.decision_function(X_te)
else:
    y_score = y_pred

reporte_clasificacion(y_te, y_pred, y_score, etiqueta=f'({MODELO} sobre {DATASET})')
print(f'\nMatriz de confusión:\n{confusion_matrix(y_te, y_pred)}')

In [ ]:
# Opcional: graficar curva ROC del bloque anterior
graficar_roc(y_te, y_score, titulo=f'ROC binaria — {MODELO}/{DATASET}')

## 2. Clasificación multiclase

**Úsalo cuando:** cada muestra pertenece a **una sola clase** entre varias opciones.

**Qué hace este bloque:**

- usa `load_digits()`,
- entrena un clasificador multiclase,
- opcionalmente fuerza estrategia `OvR` u `OvO`,
- imprime accuracy, matriz de confusión y reporte por clase.

**Qué cambiar normalmente:**

- `ESTRATEGIA`
- `MODELO`

In [ ]:
ESTRATEGIA = 'nativa'    # >>> CAMBIAR: 'nativa' | 'ovr' | 'ovo'
MODELO = 'logreg'        # >>> CAMBIAR: 'sgd' | 'knn' | 'logreg' | 'rf'

data = load_digits()
X, y = data.data, data.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

base = {
    'sgd': SGDClassifier(loss='log_loss', random_state=42, max_iter=1000),
    'knn': KNeighborsClassifier(n_neighbors=5),
    'logreg': LogisticRegression(max_iter=2000),
    'rf': RandomForestClassifier(n_estimators=100, random_state=42),
}[MODELO]

if ESTRATEGIA == 'ovr':
    clf = OneVsRestClassifier(base)
elif ESTRATEGIA == 'ovo':
    clf = OneVsOneClassifier(base)
else:
    clf = base

clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(f'\nAccuracy global: {accuracy_score(y_te, y_pred):.4f}')
print(f'Matriz de confusión 10x10:\n{confusion_matrix(y_te, y_pred)}')
print(f'\nReporte detallado:\n{classification_report(y_te, y_pred, zero_division=0)}')

## 3. Clasificación multietiqueta

**Úsalo cuando:** una misma muestra puede tener **varias etiquetas activas a la vez**.

Ejemplo típico: *“¿es par?”* y *“¿es mayor o igual a 7?”* al mismo tiempo.

**Qué hace este bloque:**

- crea una matriz `Y` con varias columnas binarias,
- entrena un modelo compatible con multioutput,
- reporta accuracy por etiqueta y F1 macro.

**Qué cambiar normalmente:**

- las reglas que construyen `y_par`, `y_grande` o tus etiquetas reales,
- el `MODELO`.

In [ ]:
MODELO = 'knn'           # >>> CAMBIAR: 'knn' | 'rf' | 'sgd' | 'logreg'

data = load_digits()
X, y = data.data, data.target

y_par = (y % 2 == 0).astype(int)      # >>> CAMBIAR: etiqueta 1
y_grande = (y >= 7).astype(int)       # >>> CAMBIAR: etiqueta 2
Y = np.c_[y_par, y_grande]

X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.25, random_state=42)

if MODELO == 'knn':
    clf = KNeighborsClassifier(n_neighbors=5)
elif MODELO == 'rf':
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
elif MODELO == 'sgd':
    clf = MultiOutputClassifier(SGDClassifier(loss='log_loss', random_state=42, max_iter=1000))
elif MODELO == 'logreg':
    clf = MultiOutputClassifier(LogisticRegression(max_iter=2000))
else:
    raise ValueError(MODELO)

clf.fit(X_tr, Y_tr)
Y_pred = clf.predict(X_te)

print(f'\nForma de la salida: {Y_pred.shape}  (n_muestras, n_etiquetas)')
print(f'Accuracy por etiqueta (par):  {accuracy_score(Y_te[:, 0], Y_pred[:, 0]):.4f}')
print(f'Accuracy por etiqueta (>=7): {accuracy_score(Y_te[:, 1], Y_pred[:, 1]):.4f}')
print(f'F1 macro (multietiqueta):    {f1_score(Y_te, Y_pred, average="macro"):.4f}')

## 4. Regresión lineal

**Úsalo cuando:** quieres predecir un **valor numérico continuo** y la relación parece aproximadamente una recta.

**Qué hace este bloque:**

- carga un CSV,
- selecciona una columna `X` y una `y`,
- ajusta un `LinearRegression`,
- muestra pendiente, intercepto, R² y predicciones nuevas.

**Qué cambiar normalmente:**

- `RUTA_CSV`
- `X_PREDECIR`
- `COL_X` y `COL_Y`

In [ ]:
RUTA_CSV = RUTA_LAB3 / 'datosVentas.csv'   # >>> CAMBIAR si te dan otro CSV
X_PREDECIR = [[2026]]                      # >>> CAMBIAR: valor(es) a predecir

df = pd.read_csv(RUTA_CSV)
print(f'Columnas del CSV: {list(df.columns)}')
display(df.head())

COL_X = df.columns[0]    # >>> CAMBIAR si el examen usa otros nombres
COL_Y = df.columns[1]

X = df[[COL_X]].values
y = df[COL_Y].values

modelo = LinearRegression()
modelo.fit(X, y)

print(f'Coeficiente (pendiente): {modelo.coef_[0]:.4f}')
print(f'Intercepto              : {modelo.intercept_:.4f}')
print(f'R² sobre entrenamiento  : {modelo.score(X, y):.4f}')

y_pred = modelo.predict(X_PREDECIR)
for x_val, y_val in zip(X_PREDECIR, y_pred):
    print(f'Predicción para X={x_val[0]}: y = {y_val:.2f}')

In [ ]:
# Opcional: graficar la recta ajustada
plt.figure(figsize=(7, 5))
plt.scatter(X, y, label='datos', color='C0')
x_grid = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(x_grid, modelo.predict(x_grid), color='C3', label='regresión lineal')
plt.xlabel(COL_X)
plt.ylabel(COL_Y)
plt.title('Regresión lineal')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Regresión polinómica

**Úsalo cuando:** la salida sigue siendo continua, pero la relación con `X` tiene **curvatura**.

**Qué hace este bloque:**

- transforma `X` a potencias (`x`, `x²`, `x³`, ...),
- ajusta un `LinearRegression` sobre esas variables transformadas,
- imprime R², MSE y predicciones nuevas.

**Qué cambiar normalmente:**

- `GRADO`
- `X_PREDECIR`
- la forma de cargar `X` y `y` si en el examen te dan un CSV real.

In [ ]:
GRADO = 3
X_PREDECIR = np.array([[0], [1.5], [3], [5]])   # >>> CAMBIAR según el examen

rng = np.random.default_rng(0)
X = np.linspace(-2, 5, 100).reshape(-1, 1)
y = (0.5 * X.ravel()**3 - 2 * X.ravel()**2 + X.ravel() + 3
     + rng.normal(0, 1.5, size=X.shape[0]))

poly = PolynomialFeatures(degree=GRADO, include_bias=False)
X_poly = poly.fit_transform(X)

modelo = LinearRegression()
modelo.fit(X_poly, y)

y_hat = modelo.predict(X_poly)
print(f'Grado del polinomio : {GRADO}')
print(f'R² entrenamiento    : {r2_score(y, y_hat):.4f}')
print(f'MSE                 : {mean_squared_error(y, y_hat):.4f}')

for x_val in X_PREDECIR:
    x_poly = poly.transform(x_val.reshape(1, -1))
    print(f'Predicción X={x_val[0]:>5}: y = {modelo.predict(x_poly)[0]:.4f}')

In [ ]:
# Opcional: graficar la curva ajustada
x_grid = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
plt.figure(figsize=(7, 5))
plt.scatter(X, y, label='datos', color='C0', s=15)
plt.plot(x_grid, modelo.predict(poly.transform(x_grid)), color='C3', label=f'polinomio grado {GRADO}')
plt.title(f'Regresión polinómica grado {GRADO}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. MLP shallow con `sklearn`

**Úsalo cuando:** el enunciado menciona una **red neuronal poco profunda** o un **MLP** y puedes resolverlo con `sklearn`.

**Qué hace este bloque:**

- genera un ejemplo multietiqueta,
- aplica validación cruzada k-fold,
- entrena un `MLPClassifier`,
- calcula AUC micro en cada fold y el promedio final.

**Qué cambiar normalmente:**

- `HIDDEN_LAYERS`
- `MAX_ITER`
- `LR`
- `N_SPLITS`
- la carga de `X` y `Y` por tu dataset real.

In [ ]:
HIDDEN_LAYERS = (64, 32)   # >>> CAMBIAR: ej. (512, 256, 128)
MAX_ITER = 100
LR = 1e-3
N_SPLITS = 5

X, y = make_classification(
    n_samples=400, n_features=20, n_informative=10,
    n_classes=2, random_state=42,
)
Y = np.c_[
    (X[:, 0] > 0).astype(int),
    (X[:, 1] + X[:, 2] > 0).astype(int),
    (X[:, 3] < X[:, 4]).astype(int),
]

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
aucs = []
for fold, (tr, te) in enumerate(kf.split(X), 1):
    mlp = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation='relu',
        solver='adam',
        learning_rate_init=LR,
        max_iter=MAX_ITER,
        random_state=42,
    )
    mlp.fit(X[tr], Y[tr])
    proba = mlp.predict_proba(X[te])
    fpr, tpr, _ = roc_curve(Y[te].ravel(), proba.ravel())
    a = auc(fpr, tpr)
    aucs.append(a)
    print(f'Fold {fold}: AUC micro = {a:.4f}')

print(f'\nAUC promedio: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

## 7. MLP con Keras/TensorFlow (referencia)

**Úsalo cuando:** el examen te pida explícitamente **Keras** o **TensorFlow**.

**Importante:** esta sección es de referencia. En el archivo original se aclara que puede no ejecutarse en entornos sin TensorFlow instalado.

**Idea clave:**

- **multietiqueta** → salida `sigmoid` + `binary_crossentropy`
- **multiclase** → salida `softmax` + `categorical_crossentropy`
- **binaria** → salida `sigmoid` + `binary_crossentropy`
- **regresión** → salida lineal + `mse`

In [ ]:
import tensorflow as tf
from tensorflow import keras

N_FEATURES = 881   # >>> CAMBIAR
N_SALIDAS = 1385   # >>> CAMBIAR

modelo = keras.Sequential([
    keras.layers.Input(shape=(N_FEATURES,)),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(N_SALIDAS, activation='sigmoid'),  # 'softmax' si es multiclase
])

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',                           # cambia según el problema
    metrics=['accuracy'],
)

modelo.summary()

# historia = modelo.fit(X_train, Y_train,
#                       validation_split=0.2,
#                       epochs=200,
#                       batch_size=32,
#                       verbose=0)
# probas = modelo.predict(X_test)

## 8. Validación cruzada k-fold

**Úsalo cuando:** te pidan evaluar el modelo con varias particiones y reportar un promedio.

**Qué hace este bloque:**

- define un modelo,
- calcula `cross_val_score`,
- imprime puntajes por fold, media y desviación estándar.

**Qué cambiar normalmente:**

- `K`
- `SCORING`
- `modelo`

In [ ]:
K = 5
SCORING = 'accuracy'   # >>> CAMBIAR: 'f1', 'roc_auc', 'r2', etc.
modelo = LogisticRegression(max_iter=2000)

data = load_digits()
X, y = data.data, (data.target == 5).astype(int)

puntajes = cross_val_score(modelo, X, y, cv=K, scoring=SCORING)
print(f'Puntajes por fold: {[f"{s:.4f}" for s in puntajes]}')
print(f'Promedio: {puntajes.mean():.4f} ± {puntajes.std():.4f}')

## 9. Curva ROC + AUC

**Úsalo cuando:** el ejercicio sea de **clasificación binaria** y te pidan la curva ROC o el AUC.

**Qué hace este bloque:**

- entrena un clasificador,
- obtiene probabilidades con `predict_proba`,
- calcula `fpr`, `tpr` y `auc`.

**Importante:** para ROC necesitas **puntajes continuos**, no sólo etiquetas predichas.

In [ ]:
data = load_digits()
X, y = data.data, (data.target == 5).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
y_score = clf.predict_proba(X_te)[:, 1]

fpr, tpr, _ = roc_curve(y_te, y_score)
auc_v = auc(fpr, tpr)
print(f'AUC = {auc_v:.4f}')

In [ ]:
# Opcional: mostrar la curva ROC del bloque anterior
graficar_roc(y_te, y_score, titulo="ROC — Logistic Regression (digit '5')")

## Recomendación final para el examen

- **No ejecutes todo el notebook por costumbre**: identifica primero el tipo de ejercicio.
- **Ejecuta sólo imports + el bloque que necesites**.
- Si el enunciado pide métricas adicionales, apóyate en los **utilitarios 8 y 9**.
- Si te dan un CSV real, normalmente sólo tendrás que adaptar la parte de carga de datos y los parámetros marcados con `>>> CAMBIAR`.